# 08 — Fast Drift MPC vs Kalman

Stress-test the control stack in a **fast drift regime** where simple filtering can lag behind parameter motion.

This notebook compares:

- no control
- moving average
- scalar Kalman
- joint Kalman
- constrained MPC-lite
- oracle reference

Core question:

> When drift is fast enough, can constrained prediction outperform pure estimation?


In [ ]:
import os, json, zipfile
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(9423)

FIG_DIR = "figures/fast_drift_mpc"
RESULTS_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

PREFIX = "08_fast_drift_mpc"

def save_fig(name):
    path_png = f"{FIG_DIR}/{PREFIX}_{name}.png"
    path_pdf = f"{FIG_DIR}/{PREFIX}_{name}.pdf"
    plt.savefig(path_png, dpi=300, bbox_inches="tight")
    plt.savefig(path_pdf, bbox_inches="tight")
    print(f"saved: {path_png}")
    print(f"saved: {path_pdf}")

THRESHOLD_45 = 1 / np.sqrt(1**2 + 1**2)
print("45° CGCS threshold:", THRESHOLD_45)


## Synthetic fast-drift calibration stream

We intentionally make drift faster than previous notebooks:

- Ω has oscillatory + local acceleration structure
- B has monotonic + oscillatory offset structure
- measurements include calibration noise


In [ ]:
N = 48
blocks = np.arange(N)

# Fast drift: higher curvature and higher local velocity than Notebook 05/06/07.
omega_true = (
    0.060 * np.sin(0.34 * blocks + 0.20)
    + 0.026 * np.sin(0.87 * blocks - 0.40)
    + 0.00075 * blocks
)

B_true = (
    0.002 + 0.00145 * blocks
    + 0.010 * np.sin(0.43 * blocks + 0.15)
    + 0.006 * np.sin(0.91 * blocks - 0.20)
)

sigma_omega = 0.0040
sigma_B = 0.0060
omega_meas = omega_true + np.random.normal(0, sigma_omega, N)
B_meas = B_true + np.random.normal(0, sigma_B, N)

print("drift correlation true:", np.corrcoef(omega_true, B_true)[0,1])
print("drift correlation measured:", np.corrcoef(omega_meas, B_meas)[0,1])


## Response model and scoring

A Rabi-like response is evaluated block-by-block against the target response.

RMSE is computed at the response level, not just parameter level.


In [ ]:
tau = np.linspace(0, 10, 220)
OMEGA_BASE = 2.48
B_BASE = 0.065

# Target hardware response with true drift removed.
def rabi_response(omega_drift, b_drift):
    omega = OMEGA_BASE + omega_drift
    b = B_BASE + b_drift
    return b + 0.46 * (1 - np.cos(omega * tau))

target_response = rabi_response(0.0, 0.0)

def block_rmse(omega_cmd, b_cmd):
    errors = []
    for ot, bt, oc, bc in zip(omega_true, B_true, omega_cmd, b_cmd):
        # command cancels estimated drift; remaining error is true drift - command
        resp = rabi_response(ot - oc, bt - bc)
        errors.append(np.sqrt(np.mean((resp - target_response) ** 2)))
    return np.array(errors)

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def block_cosines(omega_cmd, b_cmd):
    vals = []
    for ot, bt, oc, bc in zip(omega_true, B_true, omega_cmd, b_cmd):
        resp = rabi_response(ot - oc, bt - bc)
        vals.append(cosine_similarity(resp, target_response))
    return np.array(vals)

def rmse(x):
    return float(np.sqrt(np.mean(np.asarray(x) ** 2)))


## Baselines: moving average and scalar Kalman

In [ ]:
def moving_average(x, window=4):
    out = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        out[i] = np.mean(x[lo:i+1])
    return out

omega_ma = moving_average(omega_meas, window=4)
B_ma = moving_average(B_meas, window=4)


def scalar_kalman(z, q=8e-4, r=1e-5, x0=0.0, P0=1.0):
    x = x0
    P = P0
    xs, Ks = [], []
    for zi in z:
        P = P + q
        K = P / (P + r)
        x = x + K * (zi - x)
        P = (1 - K) * P
        xs.append(x)
        Ks.append(K)
    return np.array(xs), np.array(Ks)

omega_scalar, K_omega = scalar_kalman(omega_meas, q=1.5e-3, r=sigma_omega**2)
B_scalar, K_B = scalar_kalman(B_meas, q=2.0e-4, r=sigma_B**2)


## Joint Kalman estimator

The joint filter treats Ω and B as a coupled two-parameter state.


In [ ]:
def joint_kalman(z_omega, z_B, q_omega=1.5e-3, q_B=2.0e-4, coupling=0.25):
    x = np.array([0.0, 0.0])
    P = np.eye(2)
    q_cross = coupling * np.sqrt(q_omega * q_B)
    Q = np.array([[q_omega, q_cross], [q_cross, q_B]])
    R = np.diag([sigma_omega**2, sigma_B**2])
    H = np.eye(2)
    xs = []
    Ps = []
    for zo, zb in zip(z_omega, z_B):
        x_pred = x
        P_pred = P + Q
        z = np.array([zo, zb])
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)
        x = x_pred + K @ (z - H @ x_pred)
        P = (np.eye(2) - K @ H) @ P_pred
        xs.append(x.copy())
        Ps.append(P.copy())
    return np.array(xs), np.array(Ps)

joint_state, joint_cov = joint_kalman(omega_meas, B_meas)
omega_joint = joint_state[:, 0]
B_joint = joint_state[:, 1]


## Fast-drift predictive controller

This MPC-lite controller uses a short velocity estimate to forecast the next command.

It is constrained by:

- maximum command step
- regularization / blending with current joint Kalman estimate


In [ ]:
def constrained_predictive_command(x, horizon=1, max_step=0.012, alpha=0.65):
    x = np.asarray(x, dtype=float)
    v = np.zeros_like(x)
    v[1:] = x[1:] - x[:-1]
    pred = x + horizon * v
    cmd = np.zeros_like(x)
    cmd[0] = x[0]
    for i in range(1, len(x)):
        desired = alpha * pred[i] + (1 - alpha) * x[i]
        step = desired - cmd[i-1]
        step = np.clip(step, -max_step, max_step)
        cmd[i] = cmd[i-1] + step
    return cmd

omega_mpc = constrained_predictive_command(omega_joint, horizon=1, max_step=0.014, alpha=0.72)
B_mpc = constrained_predictive_command(B_joint, horizon=1, max_step=0.010, alpha=0.68)

# Naive predictive command for contrast.
def naive_predictive_command(x, horizon=3):
    v = np.zeros_like(x)
    v[1:] = x[1:] - x[:-1]
    return x + horizon * v

omega_naive = naive_predictive_command(omega_joint, horizon=3)
B_naive = naive_predictive_command(B_joint, horizon=3)


## Policy table

In [ ]:
policies = {
    "none": (np.zeros(N), np.zeros(N)),
    "moving_average": (omega_ma, B_ma),
    "scalar_kalman": (omega_scalar, B_scalar),
    "joint_kalman": (omega_joint, B_joint),
    "naive_predictive": (omega_naive, B_naive),
    "fast_drift_mpc": (omega_mpc, B_mpc),
    "oracle": (omega_true, B_true),
}

summary = {}
for name, (oc, bc) in policies.items():
    errs = block_rmse(oc, bc)
    coss = block_cosines(oc, bc)
    summary[name] = {
        "mean_rmse": float(np.mean(errs)),
        "max_rmse": float(np.max(errs)),
        "omega_rmse": rmse(omega_true - oc),
        "offset_rmse": rmse(B_true - bc),
        "min_cosine": float(np.min(coss)),
        "mean_cosine": float(np.mean(coss)),
        "max_angle_deg": float(np.degrees(np.arccos(np.clip(np.min(coss), -1, 1))))
    }

ranked = sorted(summary.items(), key=lambda kv: kv[1]["mean_rmse"])
for k, v in ranked:
    print(f"{k:18s} mean RMSE={v['mean_rmse']:.5f}  max RMSE={v['max_rmse']:.5f}  min cos={v['min_cosine']:.6f}")


## Figure 1 — Policy ranking

In [ ]:
rank_names = [k for k, _ in ranked]
mean_vals = [summary[k]["mean_rmse"] for k in rank_names]
max_vals = [summary[k]["max_rmse"] for k in rank_names]

x = np.arange(len(rank_names))
w = 0.38
plt.figure(figsize=(12, 5))
plt.bar(x - w/2, mean_vals, width=w, label="mean RMSE")
plt.bar(x + w/2, max_vals, width=w, label="max RMSE")
for i, val in enumerate(mean_vals):
    plt.text(i - w/2, val, f"{val:.3f}", ha="center", va="bottom", fontsize=8)
for i, val in enumerate(max_vals):
    plt.text(i + w/2, val, f"{val:.3f}", ha="center", va="bottom", fontsize=8)
plt.xticks(x, rank_names, rotation=30, ha="right")
plt.ylabel("response RMSE")
plt.title("Fast drift: control policy ranking")
plt.legend()
save_fig("policy_ranking_summary")
plt.show()


## Figure 2 — Ω drift tracking

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(blocks, omega_true, label="true Ω drift", linewidth=2)
plt.plot(blocks, omega_meas, marker="o", linestyle="-", label="measured Ω drift", alpha=0.75)
plt.plot(blocks, omega_ma, linestyle="--", label="moving average")
plt.plot(blocks, omega_scalar, linestyle=":", linewidth=2, label="scalar Kalman")
plt.plot(blocks, omega_joint, linestyle="-.", linewidth=2, label="joint Kalman")
plt.plot(blocks, omega_mpc, linewidth=2, label="fast-drift MPC")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Ω drift estimate / command")
plt.title("Fast drift: Ω tracking and control command")
plt.legend()
save_fig("omega_tracking")
plt.show()


## Figure 3 — B drift tracking

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(blocks, B_true, label="true B drift", linewidth=2)
plt.plot(blocks, B_meas, marker="o", linestyle="-", label="measured B drift", alpha=0.75)
plt.plot(blocks, B_ma, linestyle="--", label="moving average")
plt.plot(blocks, B_scalar, linestyle=":", linewidth=2, label="scalar Kalman")
plt.plot(blocks, B_joint, linestyle="-.", linewidth=2, label="joint Kalman")
plt.plot(blocks, B_mpc, linewidth=2, label="fast-drift MPC")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("B drift estimate / command")
plt.title("Fast drift: B tracking and control command")
plt.legend()
save_fig("offset_tracking")
plt.show()


## Figure 4 — Response-level error over time

In [ ]:
plt.figure(figsize=(12, 5))
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "naive_predictive", "fast_drift_mpc", "oracle"]:
    oc, bc = policies[name]
    plt.plot(blocks, block_rmse(oc, bc), marker="o", label=name)
plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Fast drift: response-level error comparison")
plt.legend()
save_fig("response_rmse_comparison")
plt.show()


## Figure 5 — Worst-case block response comparison

In [ ]:
# worst block among practical methods, excluding no-control/oracle
worst_errors = block_rmse(omega_mpc, B_mpc)
worst_block = int(np.argmax(worst_errors))

plt.figure(figsize=(12, 5))
plt.plot(tau, target_response, label="target", linewidth=2)
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "naive_predictive", "fast_drift_mpc", "oracle"]:
    oc, bc = policies[name]
    resp = rabi_response(omega_true[worst_block] - oc[worst_block], B_true[worst_block] - bc[worst_block])
    plt.plot(tau, resp, label=name)
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Fast drift: worst-case block {worst_block}")
plt.legend()
save_fig("worst_case_block_comparison")
plt.show()


## Figure 6 — CGCS phase-lock stability

In [ ]:
plt.figure(figsize=(12, 5))
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "naive_predictive", "fast_drift_mpc", "oracle"]:
    oc, bc = policies[name]
    plt.plot(blocks, block_cosines(oc, bc), marker="o", label=name)
plt.axhline(THRESHOLD_45, linestyle="--", label="45° threshold")
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("Fast drift: CGCS phase-lock stability")
plt.legend()
save_fig("cgcs_stability_comparison")
plt.show()


## Figure 7 — Horizon sweep

Fast-drift MPC only helps at short horizons. Long horizons over-predict and reintroduce instability.


In [ ]:
horizons = [0, 1, 2, 3, 4, 5, 8]
horizon_rmse = []
for H in horizons:
    om = constrained_predictive_command(omega_joint, horizon=H, max_step=0.014, alpha=0.72)
    bm = constrained_predictive_command(B_joint, horizon=H, max_step=0.010, alpha=0.68)
    horizon_rmse.append(float(np.mean(block_rmse(om, bm))))

best_idx = int(np.argmin(horizon_rmse))
best_H = horizons[best_idx]

plt.figure(figsize=(10, 5))
plt.plot(horizons, horizon_rmse, marker="o")
plt.axvline(1, linestyle="--", label="current H = 1")
plt.axvline(best_H, linestyle=":", label=f"best H = {best_H}")
plt.xlabel("prediction horizon H")
plt.ylabel("mean response RMSE")
plt.title("Fast drift: MPC horizon sweep")
plt.legend()
save_fig("horizon_sweep")
plt.show()

print("best horizon:", best_H, "best mean RMSE:", horizon_rmse[best_idx])


## Figure 8 — Command-bound sweep

The command radius controls the tradeoff between rapid correction and overshoot.


In [ ]:
bounds = np.array([0.003, 0.006, 0.010, 0.014, 0.020, 0.030])
bound_rmse = []
for bound in bounds:
    om = constrained_predictive_command(omega_joint, horizon=1, max_step=bound, alpha=0.72)
    bm = constrained_predictive_command(B_joint, horizon=1, max_step=0.75 * bound, alpha=0.68)
    bound_rmse.append(float(np.mean(block_rmse(om, bm))))

best_bidx = int(np.argmin(bound_rmse))
best_bound = float(bounds[best_bidx])

plt.figure(figsize=(10, 5))
plt.plot(bounds, bound_rmse, marker="o")
plt.axvline(0.014, linestyle="--", label="current Ω bound = 0.014")
plt.axvline(best_bound, linestyle=":", label=f"best Ω bound = {best_bound:.3f}")
plt.xlabel("allowed ΔΩ command radius")
plt.ylabel("mean response RMSE")
plt.title("Fast drift: command-bound sweep")
plt.legend()
save_fig("command_bound_sweep")
plt.show()

print("best bound:", best_bound, "best mean RMSE:", bound_rmse[best_bidx])


## Save results

In [ ]:
summary_payload = {
    "notebook": "08_fast_drift_mpc_vs_kalman.ipynb",
    "blocks": int(N),
    "threshold_45": float(THRESHOLD_45),
    "noise": {"sigma_omega": sigma_omega, "sigma_B": sigma_B},
    "best_horizon": int(best_H),
    "best_horizon_mean_rmse": float(horizon_rmse[best_idx]),
    "best_command_bound": float(best_bound),
    "best_command_bound_mean_rmse": float(bound_rmse[best_bidx]),
    "policy_summary": summary,
}

json_path = f"{RESULTS_DIR}/fast_drift_mpc_summary.json"
with open(json_path, "w") as f:
    json.dump(summary_payload, f, indent=2)

print(f"saved: {json_path}")


## Create output zip

Download and add extracted files to:

- `figures/fast_drift_mpc/`
- `results/fast_drift_mpc_summary.json`


In [ ]:
zip_name = "fast_drift_mpc_outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files_in_dir in os.walk(FIG_DIR):
        for filename in files_in_dir:
            path = os.path.join(root, filename)
            z.write(path, path)
    z.write(json_path, json_path)

print(f"created: {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Not running in Colab. Download the zip from the local runtime files panel.")
